In [0]:
import sys
import os
import json

project_root = os.path.abspath(os.path.join(os.getcwd(), '../..'))

if project_root not in sys.path:
    sys.path.append(project_root)

from modules.utils.date import get_target_yyyymm
from pyspark.sql.functions import lit
from pyspark.sql.functions import lit, explode

In [0]:
# Get last downloaded stations data of given city
# 1. Construct volume path based on city/system and target month.
# 2. Load stations JSON data for current month and add city column.
# 3. Append data to bronze table for the city.
months_ago = int(dbutils.widgets.get("months_ago"))
target_month = get_target_yyyymm(months_ago=months_ago)
city = dbutils.widgets.get("city")
city_dict = json.loads(dbutils.widgets.get(city))

volume_path: str = f"/Volumes/bikes/00_landing/data_sources/{city_dict["system"]}/stations/{target_month}"

In [0]:
df = spark.read.format("json").load(f"{volume_path}").\
        select(explode("data.stations").alias("stations")).\
        select("stations.*").\
        withColumn("city", lit(city_dict["city"]))

In [0]:
df.write.mode("overwrite").saveAsTable(f"bikes.01_bronze.{city_dict["city_id"]}_station_lookup_raw")